In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Salting-AQE-ON") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .getOrCreate()

print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Skew Join:", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/02 16:57:26 WARN Utils: Your hostname, Ameys-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.190 instead (on interface en0)
26/04/02 16:57:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 16:57:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/02 16:57:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/02 16:57:27 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


AQE: true
Skew Join: true


In [2]:
hot_key_rows = 10_000_000
normal_rows = 200_000

hot_df = (
    spark.range(hot_key_rows)
    .withColumn("user_id", lit(101))
    .withColumn("txn_id", col("id"))
    .withColumn("amount", (rand() * 1000).cast("int"))
    .drop("id")
)

normal_df = (
    spark.range(normal_rows)
    .withColumn("user_id", (col("id") % 5000 + 1000).cast("int"))
    .withColumn("txn_id", col("id"))
    .withColumn("amount", (rand() * 1000).cast("int"))
    .drop("id")
)

transactions_df = hot_df.union(normal_df)

users_df = (
    spark.range(7000)
    .withColumnRenamed("id", "user_id")
    .withColumn("segment", when(col("user_id") == 101, "VIP").otherwise("REGULAR"))
)

In [3]:
joined_df = transactions_df.join(users_df, on="user_id", how="inner")
joined_df.explain(True)
joined_df.count()

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [user_id])
:- Union false, false
:  :- Project [user_id#1, txn_id#2L, amount#3]
:  :  +- Project [id#0L, user_id#1, txn_id#2L, cast((rand(869730126064501321) * cast(1000 as double)) as int) AS amount#3]
:  :     +- Project [id#0L, user_id#1, id#0L AS txn_id#2L]
:  :        +- Project [id#0L, 101 AS user_id#1]
:  :           +- Range (0, 10000000, step=1, splits=Some(8))
:  +- Project [user_id#5, txn_id#6L, amount#7]
:     +- Project [id#4L, user_id#5, txn_id#6L, cast((rand(-6244415387332467340) * cast(1000 as double)) as int) AS amount#7]
:        +- Project [id#4L, user_id#5, id#4L AS txn_id#6L]
:           +- Project [id#4L, cast(((id#4L % cast(5000 as bigint)) + cast(1000 as bigint)) as int) AS user_id#5]
:              +- Range (0, 200000, step=1, splits=Some(8))
+- Project [user_id#9L, CASE WHEN (user_id#9L = cast(101 as bigint)) THEN VIP ELSE REGULAR END AS segment#10]
   +- Project [id#8L AS user_id#9L]
      +- Range (0, 7000, st

10200000

In [4]:
from pyspark.sql.functions import spark_partition_id

transactions_df_with_pid = transactions_df.withColumn(
    "partition_id",
    spark_partition_id()
)

transactions_df_with_pid.groupBy("user_id", "partition_id") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, False)

+-------+------------+-------+
|user_id|partition_id|count  |
+-------+------------+-------+
|101    |0           |1250000|
|101    |1           |1250000|
|101    |2           |1250000|
|101    |3           |1250000|
|101    |4           |1250000|
|101    |5           |1250000|
|101    |6           |1250000|
|101    |7           |1250000|
|1001   |8           |5      |
|1007   |8           |5      |
|1012   |8           |5      |
|1020   |8           |5      |
|1022   |8           |5      |
|1026   |8           |5      |
|1047   |8           |5      |
|1051   |8           |5      |
|1056   |8           |5      |
|1074   |8           |5      |
|1097   |8           |5      |
|1106   |8           |5      |
+-------+------------+-------+
only showing top 20 rows


In [5]:
users_df.show(5)

+-------+-------+
|user_id|segment|
+-------+-------+
|      0|REGULAR|
|      1|REGULAR|
|      2|REGULAR|
|      3|REGULAR|
|      4|REGULAR|
+-------+-------+
only showing top 5 rows


In [9]:
users_df.withColumn(
    'rand',
    (rand()*5).cast("int")
).show()

+-------+-------+----+
|user_id|segment|rand|
+-------+-------+----+
|      0|REGULAR|   1|
|      1|REGULAR|   5|
|      2|REGULAR|   0|
|      3|REGULAR|   1|
|      4|REGULAR|   2|
|      5|REGULAR|   4|
|      6|REGULAR|   4|
|      7|REGULAR|   4|
|      8|REGULAR|   1|
|      9|REGULAR|   2|
|     10|REGULAR|   1|
|     11|REGULAR|   2|
|     12|REGULAR|   4|
|     13|REGULAR|   3|
|     14|REGULAR|   1|
|     15|REGULAR|   0|
|     16|REGULAR|   0|
|     17|REGULAR|   5|
|     18|REGULAR|   1|
|     19|REGULAR|   4|
+-------+-------+----+
only showing top 20 rows


In [10]:
hot_product_rows = 4_000_000
normal_rows = 300_000

hot_orders = (
    spark.range(hot_product_rows)
    .withColumn("product_id", lit(999))  # HOT KEY
    .withColumn("order_id", col("id"))
    .withColumn("quantity", (rand() * 5 + 1).cast("int"))
    .drop("id")
)

normal_orders = (
    spark.range(normal_rows)
    .withColumn("product_id", (col("id") % 2000 + 100).cast("int"))
    .withColumn("order_id", col("id"))
    .withColumn("quantity", (rand() * 5 + 1).cast("int"))
    .drop("id")
)

orders_df = hot_orders.union(normal_orders)

In [11]:
products_df = (
    spark.range(3000)
    .withColumnRenamed("id", "product_id")
    .withColumn("category", when(col("product_id") == 999, "TRENDING").otherwise("NORMAL"))
)

In [12]:
orders_df.show(5)

+----------+--------+--------+
|product_id|order_id|quantity|
+----------+--------+--------+
|       999|       0|       2|
|       999|       1|       3|
|       999|       2|       1|
|       999|       3|       4|
|       999|       4|       3|
+----------+--------+--------+
only showing top 5 rows


In [29]:
orders_salted = orders_df.withColumn(
    "salt",
    when(
        col("product_id") == trending,
        (rand() * salt_factor).cast("int")
    ).otherwise(lit(0))
)

hot_products = products_df.filter(col('product_id')==999).withColumn(
    'salt',
    explode(array([lit(i) for i in range(5)]))
)

cold_products = products_df.filter(col('product_id')!=999).withColumn(
    'salt',
    lit(0)
)

products_final = hot_products.union(cold_products)

In [30]:
joined_salted = orders_salted.join(
    products_final,
    on=["product_id", "salt"],
    how="inner"
)

In [31]:
from pyspark.sql.functions import spark_partition_id, count

orders_salted.withColumn("pid", spark_partition_id()) \
    .groupBy("pid") \
    .agg(count("*").alias("rows")) \
    .orderBy("pid") \
    .show(100, False)

+---+------+
|pid|rows  |
+---+------+
|0  |500000|
|1  |500000|
|2  |500000|
|3  |500000|
|4  |500000|
|5  |500000|
|6  |500000|
|7  |500000|
|8  |37500 |
|9  |37500 |
|10 |37500 |
|11 |37500 |
|12 |37500 |
|13 |37500 |
|14 |37500 |
|15 |37500 |
+---+------+



In [32]:
from pyspark.sql.functions import *

# BIG TABLE (skewed)
hot_rows = 3_000_000
normal_rows = 200_000

hot_deliveries = (
    spark.range(hot_rows)
    .withColumn("restaurant_id", lit(42))  # HOT KEY
    .withColumn("delivery_id", col("id"))
    .withColumn("driver_id", (rand()*1000).cast("int"))
    .withColumn("order_amount", (rand()*50 + 10).cast("int"))
    .drop("id")
)

normal_deliveries = (
    spark.range(normal_rows)
    .withColumn("restaurant_id", (col("id") % 500 + 10).cast("int"))
    .withColumn("delivery_id", col("id"))
    .withColumn("driver_id", (rand()*1000).cast("int"))
    .withColumn("order_amount", (rand()*50 + 10).cast("int"))
    .drop("id")
)

deliveries_df = hot_deliveries.union(normal_deliveries)

In [33]:
# SMALL TABLE
restaurants_df = (
    spark.range(1000)
    .withColumnRenamed("id", "restaurant_id")
    .withColumn("city", when(col("restaurant_id") == 42, "NYC").otherwise("SF"))
    .withColumn("cuisine", when(col("restaurant_id") == 42, "TRENDING").otherwise("NORMAL"))
)

In [34]:
salt_factor = 5
deliveries_salted = deliveries_df.withColumn(
    'salt',
    (rand()*salt_factor).cast('int')
)

deliveries_salted.show()

+-------------+-----------+---------+------------+----+
|restaurant_id|delivery_id|driver_id|order_amount|salt|
+-------------+-----------+---------+------------+----+
|           42|          0|      910|          10|   2|
|           42|          1|      391|          42|   2|
|           42|          2|      109|          39|   2|
|           42|          3|      765|          26|   3|
|           42|          4|      524|          12|   1|
|           42|          5|      157|          54|   2|
|           42|          6|      420|          49|   1|
|           42|          7|      447|          29|   0|
|           42|          8|       61|          37|   4|
|           42|          9|       12|          23|   1|
|           42|         10|      369|          27|   4|
|           42|         11|      871|          45|   2|
|           42|         12|       81|          44|   3|
|           42|         13|      118|          55|   4|
|           42|         14|      741|          2

In [38]:


salt_factor = 5
deliveries_salted = deliveries_df.withColumn(
    'salt',
    when(col('restaurant_id')==42,(rand()*salt_factor).cast('int')).otherwise(lit(0))
)
hot_keys = restaurants_df.filter(col('restaurant_id')==42).withColumn('salt',
                                                                      explode(array([lit(i) for i in range(salt_factor)]))
                                                                    )

cold_keys = restaurants_df.filter(col('restaurant_id')!=42).withColumn('salt',
                                                                      lit(0)
                                                                    )

restaurant_final = hot_keys.union(cold_keys)

# restaurant_final.show()



dadus = deliveries_salted.join(
    restaurant_final,
    on = ['restaurant_id','salt'],
    how='inner'
)

dadus.show()

+-------------+----+-----------+---------+------------+----+-------+
|restaurant_id|salt|delivery_id|driver_id|order_amount|city|cuisine|
+-------------+----+-----------+---------+------------+----+-------+
|           10|   0|       1000|      758|          51|  SF| NORMAL|
|           10|   0|       1500|      138|          41|  SF| NORMAL|
|           10|   0|       5500|      827|          44|  SF| NORMAL|
|           10|   0|       9500|      986|          37|  SF| NORMAL|
|           10|   0|      13500|      312|          41|  SF| NORMAL|
|           10|   0|      14000|      464|          41|  SF| NORMAL|
|           10|   0|      15000|      753|          18|  SF| NORMAL|
|           10|   0|      20000|      871|          19|  SF| NORMAL|
|           10|   0|      21500|      788|          29|  SF| NORMAL|
|           10|   0|      23000|      227|          10|  SF| NORMAL|
|           10|   0|      24000|      142|          33|  SF| NORMAL|
|           10|   0|      24500|  

In [40]:
from pyspark.sql.functions import *

hot_rows = 3_000_000
normal_rows = 300_000

hot_clicks = (
    spark.range(hot_rows)
    .withColumn("region", lit("WEST"))
    .withColumn("click_id", col("id"))
    .withColumn("user_id", (rand() * 100000).cast("int"))
    .withColumn("device_type", when(rand() > 0.5, "mobile").otherwise("web"))
    .drop("id")
)

normal_clicks = (
    spark.range(normal_rows)
    .withColumn(
        "region",
        when(col("id") % 4 == 0, "EAST")
        .when(col("id") % 4 == 1, "NORTH")
        .when(col("id") % 4 == 2, "SOUTH")
        .otherwise("CENTRAL")
    )
    .withColumn("click_id", col("id"))
    .withColumn("user_id", (rand() * 100000).cast("int"))
    .withColumn("device_type", when(rand() > 0.5, "mobile").otherwise("web"))
    .drop("id")
)

clicks_df = hot_clicks.union(normal_clicks)

In [41]:
regions_df = spark.createDataFrame(
    [
        ("WEST", "NA", "tier_1"),
        ("EAST", "NA", "tier_2"),
        ("NORTH", "NA", "tier_3"),
        ("SOUTH", "NA", "tier_2"),
        ("CENTRAL", "NA", "tier_3")
    ],
    ["region", "super_region", "priority_tier"]
)

In [45]:
salt_factor = 6

clicks_salted = clicks_df.withColumn(
    'salt',
    when(col('region')=='WEST',(rand()*salt_factor).cast('int')).otherwise(lit(0))
)

hot_key = regions_df.filter(col('region')=='WEST').withColumn(
    'salt',
    explode(array([lit(i) for i in range(salt_factor)]))
)

cold_key = regions_df.filter(col('region')!='WEST').withColumn(
    'salt',
    lit(0)
)

region_final = hot_key.union(cold_key)


joined_df = clicks_salted.join(
    region_final,
    on=['region','salt'],
    how='inner'
)

In [43]:
region_final.show()

+-------+------------+-------------+----+
| region|super_region|priority_tier|salt|
+-------+------------+-------------+----+
|   WEST|          NA|       tier_1|   0|
|   WEST|          NA|       tier_1|   1|
|   WEST|          NA|       tier_1|   2|
|   WEST|          NA|       tier_1|   3|
|   WEST|          NA|       tier_1|   4|
|   WEST|          NA|       tier_1|   5|
|   EAST|          NA|       tier_2|   0|
|  NORTH|          NA|       tier_3|   0|
|  SOUTH|          NA|       tier_2|   0|
|CENTRAL|          NA|       tier_3|   0|
+-------+------------+-------------+----+

